Identificar produtos populares é extremamente importante para as empresas de comércio eletrônico! Produtos populares geram mais receita e, portanto, desempenham um papel fundamental no controle de estoque.

Você foi solicitado a dar suporte a uma livraria online, construindo um modelo para prever se um livro será popular ou não. Eles forneceram um extenso conjunto de dados contendo informações sobre todos os livros que venderam, incluindo:

price (preço)

popularity (popularidade – variável alvo)

review/summary (resumo da avaliação)

review/text (texto da avaliação)

review/helpfulness (utilidade da avaliação)

authors (autores)

categories (categorias)

Você precisará construir um modelo que preveja se um livro será classificado como popular ou não.

Eles têm grandes expectativas, por isso estabeleceram uma meta de, no mínimo, 70% de acurácia! Você está livre para usar quantas características quiser e precisará realizar engenharia de características (feature engineering) para atingir este nível de desempenho

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/content/books.csv')
df.head()

,title,price,review/helpfulness,review/summary,review/text,description,authors,categories,popularity
0,We Band of Angels: The Untold Story of America...,10.88,2/3,A Great Book about women in WWII,I have alway been a fan of fiction books set i...,"In the fall of 1941, the Philippines was a gar...",'Elizabeth Norman','History',Unpopular
1,Prayer That Brings Revival: Interceding for Go...,9.35,0/0,Very helpful book for church prayer groups and...,Very helpful book to give you a better prayer ...,"In Prayer That Brings Revival, best-selling au...",'Yong-gi Cho','Religion',Unpopular
2,The Mystical Journey from Jesus to Christ,24.95,17/19,Universal Spiritual Awakening Guide With Some ...,The message of this book is to find yourself a...,THE MYSTICAL JOURNEY FROM JESUS TO CHRIST Disc...,'Muata Ashby',"'Body, Mind & Spirit'",Unpopular
3,Death Row,7.99,0/1,Ben Kincaid tries to stop an execution.,The hero of William Bernhardt's Ben Kincaid no...,"Upon receiving his execution date, one of the ...",'Lynden Harris','Social Science',Unpopular
4,Sound and Form in Modern Poetry: Second Editio...,32.50,18/20,good introduction to modern prosody,There's a lot in this book which the reader wi...,An updated and expanded version of a classic a...,"'Harvey Seymour Gross', 'Robert McDowell'",'Poetry',Unpopular


## Trantado dados

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15719 entries, 0 to 15718
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               15719 non-null  object 
 1   price               15719 non-null  float64
 2   review/helpfulness  15719 non-null  object 
 3   review/summary      15718 non-null  object 
 4   review/text         15719 non-null  object 
 5   description         15719 non-null  object 
 6   authors             15719 non-null  object 
 7   categories          15719 non-null  object 
 8   popularity          15719 non-null  object 
dtypes: float64(1), object(8)
memory usage: 1.1+ MB


Transformando as colunas que são do tipo string em número

In [7]:
from sklearn.preprocessing import LabelEncoder

columns_string = ['title', 'review/helpfulness', 'review/summary', 'review/text', 'description', 'authors', 'categories', 'popularity']
encoder = LabelEncoder()

for col in columns_string:
    df[col] = encoder.fit_transform(df[col].astype(str))

df.head()


,title,price,review/helpfulness,review/summary,review/text,description,authors,categories,popularity
0,6689,10.88,346,336,3740,3182,1770,205,1
1,4016,9.35,0,9935,11394,3067,6418,280,1
2,5801,24.95,267,9728,8650,4956,4489,62,1
3,1451,7.99,1,2042,8601,6361,3908,293,1
4,4660,32.50,286,10970,8963,1241,2318,272,1


normalizando dados

In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df_scaler = scaler.fit_transform(df[df.columns])

df[df.columns] = pd.DataFrame(df_scaler)
df.head()

,title,price,review/helpfulness,review/summary,review/text,description,authors,categories,popularity
0,0.957761,0.242335,0.340216,0.028829,0.301394,0.458766,0.274589,0.657051,1.0
1,0.575029,0.204807,0.000000,0.852424,0.918205,0.442186,0.995656,0.897436,1.0
2,0.830613,0.587442,0.262537,0.834663,0.697075,0.714533,0.696401,0.198718,1.0
3,0.207761,0.171450,0.000983,0.175204,0.693126,0.917099,0.606267,0.939103,1.0
4,0.667239,0.772627,0.281219,0.941227,0.722298,0.178922,0.359603,0.871795,1.0


Dividindo os dados

In [10]:
y = df['popularity']
X = df.drop(columns='popularity')

,title,price,review/helpfulness,review/summary,review/text,description,authors,categories
0,0.957761,0.242335,0.340216,0.028829,0.301394,0.458766,0.274589,0.657051
1,0.575029,0.204807,0.000000,0.852424,0.918205,0.442186,0.995656,0.897436
2,0.830613,0.587442,0.262537,0.834663,0.697075,0.714533,0.696401,0.198718
3,0.207761,0.171450,0.000983,0.175204,0.693126,0.917099,0.606267,0.939103
4,0.667239,0.772627,0.281219,0.941227,0.722298,0.178922,0.359603,0.871795


## Treinando o modelo

Esse problema se enquadra bem em uma classificação binária e podemos utilizar o RadomForest para resolver ele

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Dividindo os dados

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Treinando o modelo

In [18]:
modelo = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight='balanced'
)

modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

Avaliando o modelo

In [19]:
print("Acurácia:", accuracy_score(y_test, y_pred))
print("Relatório de Classificação: ", classification_report(y_test, y_pred))
print("Matriz de Confusão: ", confusion_matrix(y_test, y_pred))

Acurácia: 0.7006997455470738
Relatório de Classificação:                precision    recall  f1-score   support

         0.0       0.65      0.19      0.30      1032
         1.0       0.71      0.95      0.81      2112

    accuracy                           0.70      3144
   macro avg       0.68      0.57      0.55      3144
weighted avg       0.69      0.70      0.64      3144

Matriz de Confusão:  [[ 200  832]
 [ 109 2003]]
